# 02c — Fine-tune with Databricks Mosaic AI Model Training

**Purpose:** The lowest-code path. Submits a **managed** supervised fine-tuning run via the Databricks Mosaic AI Model Training API (`databricks-genai`). No cluster/GPU management — Databricks provisions compute, runs training, and registers the result to Unity Catalog.

Consumes the JSONL dataset from `01`. Same `BASE_MODEL` idea, but the base must be one of the API's supported models.

> **Caveat:** Availability of Mosaic AI Model Training varies by workspace region and entitlement. This path is managed and less transparent than `02a`/`02b` — you get a registered model, not the training loop. Check the [supported models list](https://docs.databricks.com/en/large-language-models/foundation-model-training/index.html) before running.

In [ ]:
%pip install -q databricks-genai mlflow
dbutils.library.restartPython()

In [ ]:
# ─────────────────────────────────────────────────────────────
# CONFIGURATION
# ─────────────────────────────────────────────────────────────
# Must be a model supported by Mosaic AI Model Training (not arbitrary HF ids).
BASE_MODEL = "meta-llama/Llama-3.2-3B-Instruct"

CATALOG = "main"
SCHEMA  = "otel_finetuning"
VOLUME  = f"/Volumes/{CATALOG}/{SCHEMA}/finetune"
TRAIN_DATA_PATH = f"{VOLUME}/train.jsonl"
EVAL_DATA_PATH  = f"{VOLUME}/eval.jsonl"

REGISTER_TO = f"{CATALOG}.{SCHEMA}.otel_sft_model"   # UC-registered output model
TRAINING_DURATION = "3ep"                             # epochs, e.g. "3ep"
LEARNING_RATE = "2e-5"

## Submit the managed fine-tuning run

In [ ]:
from databricks.model_training import foundation_model as fm

run = fm.create(
    model=BASE_MODEL,
    task_type="CHAT_COMPLETION",          # our data is ChatML {"messages": [...]}
    train_data_path=TRAIN_DATA_PATH,
    eval_data_path=EVAL_DATA_PATH,
    register_to=REGISTER_TO,
    training_duration=TRAINING_DURATION,
    learning_rate=LEARNING_RATE,
)
print("Submitted run:", run.name)

## Poll status

In [ ]:
from databricks.model_training import foundation_model as fm

# Re-run this cell to refresh. Terminal states: COMPLETED / FAILED.
current = fm.get(run.name)
print("Status:", current.status)
display(fm.get_events(run.name))

## Result

On completion, the fine-tuned model is registered at `REGISTER_TO` in Unity Catalog. Use that model name in `03_evaluate.ipynb` (or deploy it to a serving endpoint). No adapter files to manage — the managed run handles weights and registration.

In [ ]:
print(f"When COMPLETED, the fine-tuned model is registered as: {REGISTER_TO}")
print("Point 03_evaluate.ipynb at this UC model to compare against the base model.")